In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import f1_score, classification_report
from autogluon.tabular import TabularPredictor

In [7]:
df = pd.read_csv("merged_data_combined.csv")
print(f"Shape of the dataframe: {df.shape}")
print("Head of the dataframe:")
display(df.head())

Shape of the dataframe: (1992258, 5)
Head of the dataframe:


,PC1,PC2,PC3,CME,HALO
0,-0.9736,-0.1647,1.1289,0,0
1,-1.1599,0.0643,1.0100,0,0
2,-1.3350,-0.2514,1.1276,0,0
3,-1.3588,-0.1939,1.0940,0,0
4,-0.9674,0.1175,0.9423,0,0


In [4]:
def load_and_preprocess(csv_path):
    """Load and preprocess the data"""
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"Initial shape: {df.shape}")

    if 'epoch_for_cdf_mod' in df.columns:
        df['epoch_for_cdf_mod'] = pd.to_datetime(df['epoch_for_cdf_mod'], errors='coerce')

    df.replace(-1e31, np.nan, inplace=True)
    df = df.sort_values('epoch_for_cdf_mod')
    df.dropna(thresh=len(df.columns) * 0.7, inplace=True)
    df = df.ffill().bfill()
    print(f"After cleaning: {df.shape}")

    if 'epoch_for_cdf_mod' in df.columns:
        df['hour'] = df['epoch_for_cdf_mod'].dt.hour
        df['weekday'] = df['epoch_for_cdf_mod'].dt.weekday

    columns_to_remove = ['epoch_for_cdf_mod', 'spacecraft_xpos', 'spacecraft_ypos', 'spacecraft_zpos']
    features = [col for col in df.columns if col not in columns_to_remove + ['CME', 'HALO']]
    print(f"Feature columns: {features}")

    return df, features


In [9]:
def run_autogluon(csv_path):
    df, features = load_and_preprocess(csv_path)

    # Split data
    split_time = df['epoch_for_cdf_mod'].quantile(0.8)
    train = df[df['epoch_for_cdf_mod'] < split_time]
    test = df[df['epoch_for_cdf_mod'] >= split_time]

    print("Training CME model...")
    predictor_cme = TabularPredictor(label='CME').fit(train[features + ['CME']])
    pred_cme = predictor_cme.predict(test[features])
    print("AutoGluon CME F1:", f1_score(test['CME'], pred_cme, average='weighted'))

    print("Training HALO model...")
    predictor_halo = TabularPredictor(label='HALO').fit(train[features + ['HALO']])
    pred_halo = predictor_halo.predict(test[features])
    print("AutoGluon HALO F1:", f1_score(test['HALO'], pred_halo, average='weighted'))

if __name__ == "__main__":
    run_autogluon("merged_data_combined.csv")

Loading data from merged_data_combined.csv...
Initial shape: (5130185, 15)


No path specified. Models will be saved in: "AutogluonModels/ag-20250709_153610"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.5
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 23.5.0: Wed May  1 20:12:58 PDT 2024; root:xnu-10063.121.3~5/RELEASE_ARM64_T6000
CPU Count:          8
Memory Avail:       3.33 GB / 16.00 GB (20.8%)
Disk Space Avail:   120.87 GB / 460.43 GB (26.3%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accura

After cleaning: (2041148, 15)
Feature columns: ['proton_density', 'proton_bulk_speed', 'proton_xvelocity', 'proton_yvelocity', 'proton_zvelocity', 'proton_thermal', 'alpha_density', 'alpha_bulk_speed', 'alpha_thermal', 'hour', 'weekday']
Training CME model...


	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "/Users/vishrutgupta/Desktop/ISRO_bah_2025/AutogluonModels/ag-20250709_153610"
Train Data Rows:    1632918
Train Data Columns: 11
Label Column:       CME
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [0, 1]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:       

[1000]	valid_set's binary_error: 0.144091
[2000]	valid_set's binary_error: 0.104287
[3000]	valid_set's binary_error: 0.0840784
[4000]	valid_set's binary_error: 0.070545
[5000]	valid_set's binary_error: 0.0620331
[6000]	valid_set's binary_error: 0.0571953



KeyboardInterrupt

